# EDA: домены, разметка, JPEG-метаданные

Разбор данных соревнования AIIJC 2026 (AIC), на которых обучается финальное
решение `disentangle_b2_li760_r8_all_data_hard_pixel_ft`. Работает **без GPU**
(только CPU: чтение JPEG-заголовков, парсинг имён файлов, pandas). Требует
распакованные данные соревнования по пути из `.env` (`AIIJC_DATA_PATH`) и
собранный протокол `runs/validation_protocol_20260908/protocol` (входит в
репозиторий, пересчитывать не нужно).

Часть подхода (разбор имени файла на домен/генератор/group_id, метод чтения
таблицы квантования JPEG без декодирования пиксельных данных) отражает более
раннюю разведку в соседнем инструментарии experimental-tools-beliy-russak
(`aic/data.py::parse_domain`, `aic/forensic.py::luma_qtable`) — здесь всё
переписано на функции, которые уже используются самим пайплайном
(`src/eval/metadata.py`, `src/forensic/jpeg.py`), без кросс-репо зависимости.


In [ ]:
# %% Установка минимального набора зависимостей для офлайн-EDA (без torch/timm).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
               "numpy", "pandas", "pillow", "pyarrow", "matplotlib"], check=True)


In [ ]:
# %% Корень проекта и протокол.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "src").is_dir():
    raise RuntimeError("Откройте ноутбук из корня репозитория или из notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd

from src.eval.protocol import EvaluationProtocol

PROTOCOL_PATH = ROOT / "runs" / "validation_protocol_20260908" / "protocol"
protocol = EvaluationProtocol.load(PROTOCOL_PATH)
rows = protocol.all_training_rows()  # train + development + holdout, включая originals
print("Всего размеченных строк (train+dev+holdout, вкл. originals):", len(rows))
print("Роли:", rows.role.value_counts().to_dict())
print("Колонки:", list(rows.columns))


## 1. Домены: `plain` — это два разных датасета

`src/eval/metadata.py::_parse_domain` уже отделяет `coco`/`raise`/`openimages`/
`vision`/`numid`, но весь числовой `plain` попадает в одну корзину. Внутри неё
скрыты два разных источника: имена с 8-значным стемом (`plain_l8`, после снятия
12-символьного hex-префикса) и с 9-значным (`plain_l9`, у которого префикса не
было изначально). Это не переименование — при обучении/оценке стоит хотя бы
считать метрики по обеим корзинам раздельно, поэтому здесь это выделяется явно.


In [ ]:
# %% Разбиваем domain == 'plain' на plain_l8 / plain_l9 по длине стема.
rows = rows.copy()
is_plain = rows["domain"] == "plain"
rows["domain_fine"] = rows["domain"].where(~is_plain,
    rows["stem"].str.len().map(lambda n: "plain_l9" if n == 9 else "plain_l8"))

domain_counts = rows.groupby("domain_fine").agg(
    n=("stem", "size"),
    negatives=("is_negative", "sum"),
    negative_rate=("is_negative", "mean"),
    mean_area=("mask_area", "mean"),
    mean_height=("height", "mean"),
    mean_width=("width", "mean"),
).sort_values("n", ascending=False)
domain_counts


## 2. `qt_luma`: доменный маркер прямо из заголовка JPEG, без декодирования

`src.forensic.jpeg.luma_qtable` читает таблицу квантования яркости JPEG через
`PIL.Image.open(...).quantization` — это только заголовок (DQT-маркер), декодировать
пиксели не нужно, поэтому стоимость операции равна нулю GFLOPs с точки зрения
модели. Берём `Q[0, 0]` (DC-коэффициент) как скалярный признак качества/
рекомпрессии кадра и проверяем его как самостоятельный (нейросеть тут не участвует)
детектор `plain_l9`.


In [ ]:
# qt_luma по случайной подвыборке (полный проход по всем файлам необязателен для EDA).
import numpy as np

from src.data.data_workspace import DataWorkspace
from src.forensic.jpeg import luma_qtable

SAMPLE_PER_DOMAIN = 300
workspace = DataWorkspace(ROOT / "data")

sample = (rows.groupby("domain_fine", group_keys=False)
              .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_DOMAIN), random_state=42)))


def read_qt_luma_dc(rel_path: str) -> float | None:
    path = workspace.train_root / str(rel_path).replace("\\", "/")
    if not path.is_file():
        return None
    table = luma_qtable(path)
    return None if table is None else float(table[0, 0])


sample = sample.assign(qt_luma_dc=sample["chng_img_path"].map(read_qt_luma_dc))
qt_by_domain = sample.groupby("domain_fine")["qt_luma_dc"].agg(["mean", "median", "count"])
qt_by_domain


In [ ]:
# qt_luma_dc >= 8 как самостоятельный детектор plain_l9 (без модели, без GFLOPs).
labelled = sample.dropna(subset=["qt_luma_dc"])
predicted_l9 = labelled["qt_luma_dc"] >= 8
actual_l9 = labelled["domain_fine"] == "plain_l9"

tp = int((predicted_l9 & actual_l9).sum())
fp = int((predicted_l9 & ~actual_l9).sum())
fn = int((~predicted_l9 & actual_l9).sum())
precision = tp / (tp + fp) if (tp + fp) else float("nan")
recall = tp / (tp + fn) if (tp + fn) else float("nan")
print(f"qt_luma_dc >= 8 как детектор plain_l9: precision={precision:.3f}, recall={recall:.3f}, tp={tp}, fp={fp}, fn={fn}")


**TODO (авторам):** впишите здесь фактические precision/recall после прогона на
полных данных (сейчас — выборка `SAMPLE_PER_DOMAIN=300` на домен, не весь train).
Если детектор надёжен (высокий precision, recall не обязателен — это диагностика,
не сабмит), его можно использовать как бесплатный (0 GFLOPs) сигнал для анализа
ошибок по под-доменам `plain`, а не как часть forward-пути модели: включать его в
модель как признак имеет смысл только если он не выдаёт информацию о разметке
(тут это метаданные самого JPEG, а не GT — использовать безопасно).


## 3. Площадь маски: корзины и доля негативов по домену


In [ ]:
# Корзины площади GT-маски и негативы по доменам.
bins = [-1e-9, 0.0, 0.01, 0.05, 0.20, 1.0]
labels = ["negative (0%)", "(0%, 1%]", "(1%, 5%]", "(5%, 20%]", "(20%, 100%]"]
rows["area_bucket"] = pd.cut(rows["mask_area"], bins=bins, labels=labels)

area_table = pd.crosstab(rows["domain_fine"], rows["area_bucket"], normalize="index").round(3)
area_table


## 4. Train vs test: вес файла (proxy сложности/рекомпрессии)

Домен для test-выборки не размечен, но имя файла разбирается тем же парсером —
можно сравнить распределение размера файла (байт) по общим доменам между train
и test, не заглядывая в test-разметку (которой не существует).


In [ ]:
# Сравнение веса файла train vs test по доменам, без использования test GT.
from src.eval.metadata import _parse_domain, _stem  # тот же парсер имён, что и в build_metadata

test_csv = workspace.test_csv.copy()
test_csv["stem"] = test_csv["img_path"].map(_stem)
test_csv["domain"] = test_csv["stem"].map(_parse_domain)
test_csv["file_size"] = test_csv["img_path"].map(
    lambda p: (workspace.test_root / str(p).replace("\\", "/")).stat().st_size
    if (workspace.test_root / str(p).replace("\\", "/")).is_file() else np.nan)

train_size = rows.assign(
    file_size=rows["chng_img_path"].map(
        lambda p: (workspace.train_root / str(p).replace("\\", "/")).stat().st_size
        if (workspace.train_root / str(p).replace("\\", "/")).is_file() else np.nan))

compare = pd.concat([
    train_size.groupby("domain")["file_size"].mean().rename("train_mean_bytes"),
    test_csv.groupby("domain")["file_size"].mean().rename("test_mean_bytes"),
], axis=1)
compare["test_over_train"] = compare["test_mean_bytes"] / compare["train_mean_bytes"]
compare


**TODO (авторам):** если `test_over_train` заметно меньше 1 для какого-то домена —
это тот самый разрыв val→test по «весу файла», который ранее наблюдался в этом
проекте (разные домены рекомпрессируются с разной агрессивностью на test). Впишите
фактические числа и решите, стоит ли добавлять `jpeg_recompression` аугментацию
агрессивнее для соответствующих доменов — сейчас в `baseline.yaml` она общая для
всех доменов (`jpeg_recompression_probability: 0.3`, `quality_range: [60, 100]`).


## 5. Потолок разметки при округлении маски (resize round-trip)

Модель предсказывает на `dataset.image_size` (760) и восстанавливает вероятности
до исходного разрешения перед бинаризацией (`src/data/geometry.py::restore_probability`).
Проверяем, сколько Dice теряется от одного только даунсемпла+апсемпла GT-маски
(без всякой модели) — это верхняя граница качества, которую сама архитектура
физически не может превысить из-за ресайза, не путать с потерями от самой модели.


In [ ]:
# Потолок round-trip: Dice(GT, resize_down_up(GT)) на подвыборке позитивов.
import cv2

def round_trip_dice(mask_path: str, size: int = 760, threshold: int = 128) -> float | None:
    path = workspace.train_root / str(mask_path).replace("\\", "/")
    if not path.is_file():
        return None
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    gt = (mask >= threshold).astype(np.float32)
    down = cv2.resize(gt, (size, size), interpolation=cv2.INTER_AREA)
    up = cv2.resize(down, (mask.shape[1], mask.shape[0]), interpolation=cv2.INTER_LINEAR)
    pred = (up >= 0.5).astype(np.float32)
    intersection = float((gt * pred).sum())
    return (2 * intersection + 1e-6) / (gt.sum() + pred.sum() + 1e-6)


positives = rows.loc[~rows["is_negative"]].groupby("domain_fine", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 100), random_state=42))
positives = positives.assign(round_trip_dice=positives["gt_path"].map(round_trip_dice))
positives.groupby("domain_fine")["round_trip_dice"].mean().sort_values()


## Итоги (переносятся в README / карточки экспериментов)

- `plain` в текущем `src/eval/metadata.py` — механическая склейка двух источников
  (`plain_l8`/`plain_l9`, см. раздел 1); при анализе ошибок и при взвешенной
  валидации (`eval.selection_small_mask_weight`) стоит смотреть на них раздельно.
- `qt_luma` (Q[0,0] из заголовка JPEG) — бесплатный (0 GFLOPs) сигнал качества/
  рекомпрессии кадра, пригодный для диагностики, но не проверенный здесь как
  признак модели.
- Разрыв train↔test по весу файла и потолок round-trip — количественные проверки,
  которые обязательно нужно перезапустить на полных данных перед тем, как делать
  выводы (эта версия ноутбука считает по случайным подвыборкам ради скорости).

**TODO (авторам):** этот ноутбук — воспроизводимый каркас. Более ранние прогоны
этого анализа (на полном датасете, с фактическими числами: потолок round-trip
≈0.998, разница train/test по весу файла ≈11-13% и т.п.) обсуждались вне этого
репозитория — если у вас сохранились те результаты, вставьте их сюда вместо
плейсхолдеров выше и переиспользуйте код ячеек для перепроверки на актуальных
данных, а не для повторного ввода чисел вручную.
